### Summarizing Mani Mama Lecture

In [ ]:
import chromadb
from langchain_community.document_loaders import YoutubeLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


DB_DIR = "./chromadb"
# Initialize your Chroma vector store
chroma_client = chromadb.PersistentClient(path=DB_DIR)

collection = chroma_client.get_or_create_collection(name="mani_mama_collection")

def load_and_store_youtube_video(video_url: str):
    loader = YoutubeLoader.from_youtube_url(
        video_url,
        add_video_info=False,
        chunk_size_seconds=60
    )
    docs = loader.load()
    print(docs[0].metadata["source"])
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=2000,
        chunk_overlap=200,
        separators=[""," ",'',' '],
        length_function=len,
    )
    splits = text_splitter.split_documents(docs)
    split_counter = 0

    # 2. Add documents to the Chroma vector store
    for split in splits:
        collection.add(documents=[split.page_content], ids=[f"{split.metadata['source']}_{split_counter}"])
        split_counter += 1



In [6]:
from youtube_transcript_api import TranscriptsDisabled, YouTubeTranscriptApi
yt_api_instance = YouTubeTranscriptApi()


# Get the exact video ID
video_id = "spQkJiYuTj0"
try:

    transcript_list = yt_api_instance.fetch(video_id)

    transcript = " ".join(chunk.text for chunk in transcript_list)
    
    print(transcript)
    print("---" * 100)

except TranscriptsDisabled:
    print("No captions available for this video.")

om offering my S namaskaram at the Lotus feet of our Guru mahasan and sidam and also to all my other gurus my Manas gur parandar Swami and all gurus I'm starting this third round of shat bhagat Gita lectures when I was in Sheri recently I sought the anugraham of both the aaras for me to go ahead and do the third round of bhagavat Gita and also the second round of both the Das upanishads and the brasas and often on I will also do some pranas which have been done already and now bhagavat Gita was the one which we chose to do first because of popular demand by a lot of people who wanted to again see bhagavat Gita and among the three pranas Pras our Prana is the dasas the 10ish and then brasas that is called nyaya Prana and Bhagavad Gita is called smti Prana all three are authoritarian texts which expound the tenets of our tradition and our vant but we are going to approach Bhagavad Gita as we done earlier from a different angle more from the vedantic angle it will not be just a literal me

In [ ]:
# Ingest all the documents for dhyana slokas and chapter 1 
video_urls = [
    "https://youtu.be/spQkJiYuTj0",
    "https://youtu.be/8JN2Ywd5x8Y",
    "https://youtu.be/BiECwNrD0yE",
    "https://youtu.be/2RAFDnej5SM",
    "https://youtu.be/_qtaRvpwEyA"
]

for url in video_urls:
    load_and_store_youtube_video(url)

In [ ]:
from langchain_ollama import ChatOllama
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_chroma import Chroma
vector_store = Chroma(collection_name="mani_mama_collection", client=chroma_client)

def query_after_getting_matched_documents(user_query, ollama_model_name="granite4.1:3b"):
    # Create a retriever from the vector store getting top 10 similar documents
    retriever = vector_store.as_retriever(collection_name="mani_mama_collection", search_type="similarity", search_kwargs={"k": 3})

    llm = ChatOllama(model=ollama_model_name, base_url=None)
    # ConversationalRetrievalChain wraps the LLM + retriever
    chain = ConversationalRetrievalChain.from_llm(llm=llm, retriever=retriever, return_source_documents=True)

    result = chain.invoke({"question": user_query, "chat_history":[]})
    print(result["answer"])
    matching_docs = result["source_documents"]
    print("Matching document IDs:")
    for doc in matching_docs:
        print('----------------------')
        print(doc)
        print('----------------------')

In [ ]:
query = "What does arjuna think of family traditions?" 
query_after_getting_matched_documents(query)

In [ ]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

ollama_model_name="granite4.1:3b"
llm = ChatOllama(model=ollama_model_name, base_url=None)

template = """
Write a concise summary of the following 

Give the summary in points
{context}
"""
prompt = ChatPromptTemplate.from_template(template)
chain = create_stuff_documents_chain(llm, prompt)
ans = chain.invoke({'context':docs})
print(ans)